# Text-to-SQL Evaluation — Football Dataset

**Approach:** LangGraph-based multi-step reasoning  
**Dataset:** Football Database (`exp_v3` schema, PostgreSQL)  

## Pipeline overview

```
Question
   │
   ▼
select_tables  ─► identify which tables are relevant
   │
   ▼
generate_sql   ─► produce a PostgreSQL SELECT statement
   │
   ▼
execute_sql    ─► run query against the DB
   │
   ├─(error, retry < 2)─► fix_sql ─► execute_sql  (retry loop)
   │
   └─(success / max retries)─► generate_answer
                                    │
                                    ▼
                               Natural-language answer
```

In [108]:
# Install required packages
%pip install -qU "sqlalchemy>=2" psycopg2-binary \
    langchain-core langchain-community langchain-ollama langchain-openai \
    langgraph python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


## 1. Database Connection

In [109]:
import os
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase

load_dotenv()

username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host     = os.getenv("DB_HOST")
port     = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

database_uri = f"postgresql://{username}:{password}@{host}:{port}/{database}"

try:
    sqldb = SQLDatabase.from_uri(database_uri, schema="exp_v3")
    print("Successfully connected to the Football Database!")
    names = sqldb.get_usable_table_names()
    print(f"Usable Tables ({len(names)}): {names}")
except Exception as e:
    print(f"Connection failed. Error: {e}")

Successfully connected to the Football Database!
Usable Tables (15): ['club', 'club_league_history', 'coach', 'coach_club_team', 'league', 'match_fact', 'national_opponent_team', 'national_team', 'player', 'player_club_team', 'player_fact', 'plays_match', 'stadium', 'world_cup', 'world_cup_result']


In [110]:
schema_snippet = sqldb.get_table_info()
print("\n--- Schema Snippet (first 2000 chars) ---")
print(schema_snippet[:2000])


--- Schema Snippet (first 2000 chars) ---

CREATE TABLE exp_v3.club (
	club_id VARCHAR NOT NULL, 
	club_name VARCHAR NOT NULL, 
	country VARCHAR, 
	found_year INTEGER, 
	CONSTRAINT club_pk PRIMARY KEY (club_id)
)

/*
3 rows from club table:
club_id	club_name	country	found_year
Q18741	Tottenham Hotspur F.C.	United Kingdom	1882
Q18708	Fulham F.C.	United Kingdom	1879
Q301973	Futbol Club Andorra Veterans	Andorra	1996
*/


CREATE TABLE exp_v3.club_league_history (
	club_id VARCHAR, 
	league_id VARCHAR, 
	start_year INTEGER, 
	end_year INTEGER, 
	CONSTRAINT club_league_history_club_id_fk FOREIGN KEY(club_id) REFERENCES exp_v3.club (club_id), 
	CONSTRAINT club_league_history_league_id_fk FOREIGN KEY(league_id) REFERENCES exp_v3.league (league_id)
)

/*
3 rows from club_league_history table:
club_id	league_id	start_year	end_year
Q1124841	Q12837728	2001	0
Q153539	Q82595	2021	2022
Q56734527	Q237181	2021	2022
*/


CREATE TABLE exp_v3.coach (
	nickname VARCHAR, 
	coach_id INTEGER NOT NULL, 
	coun

## 2. LLM Configuration

Set `LLM_PROVIDER=openai` in your `.env` to use OpenAI (requires `OPENAI_API_KEY`).  
Default: Ollama with `llama3.2:1b` (must be running locally on port 11434).
Override the model name via `OLLAMA_MODEL` or `OPENAI_MODEL`.

In [111]:
from dotenv import load_dotenv
import os

load_dotenv()

LLM_PROVIDER  = os.getenv("LLM_PROVIDER", "ollama")
OLLAMA_MODEL  = "qwen2.5-coder:14b"
OPENAI_MODEL  = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    print(f"Using OpenAI: {OPENAI_MODEL}")
else:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    print(f"Using Ollama: {OLLAMA_MODEL}")


Using Ollama: qwen2.5-coder:14b


## 3. LangGraph Pipeline

In [112]:
from typing import TypedDict, Optional, List


class TextToSQLState(TypedDict):
    question:        str
    relevant_tables: List[str]
    schema:          str
    sql_query:       str
    sql_result:      Optional[str]
    error:           Optional[str]
    retry_count:     int
    answer:          str

In [113]:
DB_SCHEMA  = "exp_v3"
ALL_TABLES = sqldb.get_usable_table_names()

# ── Step 1: Table usage guide ────────────────────────────────────────────
TABLE_GUIDE = """
Key table usage guide (read carefully before writing SQL):
- plays_match      : one row per team per match; columns: stage ('Group A', 'Semi-finals', etc.), year, team_goals, opponent_team_goals, team_id, opponent_team_id, stadium_id
- match_fact       : one row per PLAYER per match; columns: player_id, match_team_id, goal ('true'/'false'), minute — use for INDIVIDUAL player statistics
- player_fact      : players in a NATIONAL team roster; links player_id to team_id (national_team) — use for "which players played for country X in year Y"
- player_club_team : players in CLUB teams — do NOT use for national team queries
- national_team    : columns: team_id, teamname, year, goals (total goals for that team in that tournament)
- world_cup_result : tournament final results per year: winner ('true'/'false'), runner_up — NOT for individual match data
- stadium          : join with plays_match via stadium_id for stadium statistics
- world_cup        : one row per tournament year; has venue column

Important rules:
- For match groups/stages (Group A, Semi-finals etc.) → use plays_match.stage
- For "how many games won" → use plays_match comparing team_goals > opponent_team_goals
- For "which players played for [country]" → use player_fact JOIN national_team
- For "goals scored by player" → use match_fact WHERE goal = 'true'
- For "total goals by team in tournament" → use national_team.goals
- For "stadium game count" → use plays_match JOIN stadium (NOT match_fact)
- For "world cup winner/runner-up" → use world_cup_result
- When the question asks for a name AND a number (e.g. "oldest club"), SELECT both columns
- SELECT only the columns explicitly asked for — do NOT add extra columns (e.g. counts) unless the question asks for them
"""

# ── Step 2: Extended few-shot examples (8 examples) ─────────────────────
FEW_SHOT_EXAMPLES = """
-- Example 1 (easy): simple aggregation
Question: How often did Italy participate in the World Cup?
SQL: SELECT count(*) FROM exp_v3.national_team WHERE teamname = 'Italy'

-- Example 2 (easy): count teams using national_team table
Question: How many teams participated in the 2018 World Cup?
SQL: SELECT count(teamname) FROM exp_v3.national_team WHERE year = 2018

-- Example 3 (medium): group stage query using plays_match.stage
Question: Which teams were in Group B in 2014?
SQL: SELECT DISTINCT T2.teamname
     FROM exp_v3.plays_match AS T1
     JOIN exp_v3.national_team AS T2 ON T1.team_id = T2.team_id
     WHERE T1.year = 2014 AND T1.stage = 'Group B'

-- Example 4 (medium): oldest/newest with multiple columns
Question: Which is the newest club and when was it founded?
SQL: SELECT club_name, found_year FROM exp_v3.club ORDER BY found_year DESC LIMIT 1

-- Example 5 (hard): individual match wins using plays_match goals
Question: How many games did Germany win?
SQL: SELECT count(*)
     FROM exp_v3.national_team AS T1
     JOIN exp_v3.plays_match AS T2 ON T1.team_id = T2.team_id
     WHERE T1.teamname = 'Germany' AND T2.team_goals > T2.opponent_team_goals

-- Example 6 (hard): national team roster using player_fact
Question: Which players were in the Argentina squad in 2022?
SQL: SELECT DISTINCT p.player_name
     FROM exp_v3.player AS p
     JOIN exp_v3.player_fact AS pf ON p.player_id = pf.player_id
     JOIN exp_v3.national_team AS nt ON pf.team_id = nt.team_id
     WHERE nt.teamname = 'Argentina' AND nt.year = 2022

-- Example 7 (extra): list of teams — only teamname, no extra columns
Question: Show all teams who reached the final more than once.
SQL: SELECT T2.teamname
     FROM exp_v3.world_cup_result AS T1
     JOIN exp_v3.national_team AS T2 ON T1.team_id = T2.team_id
     WHERE T1.runner_up = 'true' OR T1.winner = 'true'
     GROUP BY T2.teamname HAVING COUNT(T1.year) > 1

-- Example 8 (extra): stadium game count using plays_match JOIN stadium
Question: Which stadium hosted the fewest games?
SQL: SELECT T1.stadium_name, count(*)
     FROM exp_v3.stadium AS T1
     JOIN exp_v3.plays_match AS T2 ON T1.stadium_id = T2.stadium_id
     GROUP BY T1.stadium_name ORDER BY count(*) ASC LIMIT 1
"""


def _clean_sql(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        end = -1 if lines[-1].strip() == "```" else len(lines)
        text = "\n".join(lines[1:end])
    return text.strip()


# ── Node 1: select_tables ────────────────────────────────────────────────
def select_tables(state: TextToSQLState) -> dict:
    prompt = (
        "You are a SQL expert helping select relevant database tables.\n\n"
        f"Available tables: {', '.join(ALL_TABLES)}\n\n"
        f"{TABLE_GUIDE}\n"
        f"Question: {state['question']}\n\n"
        "List ONLY the table names needed to answer this question, "
        "separated by commas. No explanation — just the table names."
    )
    response = llm.invoke(prompt)
    selected = [t.strip() for t in response.content.split(",")]
    valid = [t for t in selected if t in ALL_TABLES] or ALL_TABLES
    schema = sqldb.get_table_info(valid)
    return {"relevant_tables": valid, "schema": schema}


# ── Node 2: generate_sql ─────────────────────────────────────────────────
def generate_sql(state: TextToSQLState) -> dict:
    prompt = (
        "You are a PostgreSQL expert. Generate a valid SQL SELECT query "
        "to answer the question below.\n\n"
        f"Schema (tables in schema '{DB_SCHEMA}'):\n{state['schema']}\n\n"
        f"{TABLE_GUIDE}\n"
        "Rules:\n"
        f"- Always prefix table names with the schema: {DB_SCHEMA}.table_name\n"
        "- Return ONLY the raw SQL — no markdown, no code fences, no explanation\n"
        "- The query must be a SELECT statement\n"
        "- Use standard PostgreSQL syntax\n"
        "- SELECT only the columns explicitly asked for in the question\n\n"
        f"Here are some examples:\n{FEW_SHOT_EXAMPLES}\n"
        f"Question: {state['question']}\n\nSQL:"
    )
    response = llm.invoke(prompt)
    sql = _clean_sql(response.content)
    return {"sql_query": sql, "error": None}


# ── Node 3: execute_sql ──────────────────────────────────────────────────
def execute_sql(state: TextToSQLState) -> dict:
    try:
        result = sqldb.run(state["sql_query"])
        return {"sql_result": str(result), "error": None}
    except Exception as exc:
        return {
            "sql_result": None,
            "error": str(exc),
            "retry_count": state.get("retry_count", 0) + 1,
        }


# ── Node 4: fix_sql ──────────────────────────────────────────────────────
def fix_sql(state: TextToSQLState) -> dict:
    prompt = (
        "You are a PostgreSQL expert. A SQL query failed — fix it.\n\n"
        f"Schema (tables in schema '{DB_SCHEMA}'):\n{state['schema']}\n\n"
        f"{TABLE_GUIDE}\n"
        f"Original question: {state['question']}\n\n"
        f"Failing SQL:\n{state['sql_query']}\n\n"
        f"Error message:\n{state['error']}\n\n"
        "Rules:\n"
        f"- Always prefix table names with the schema: {DB_SCHEMA}.table_name\n"
        "- Return ONLY the corrected raw SQL — no markdown, no code fences\n\n"
        "Fixed SQL:"
    )
    response = llm.invoke(prompt)
    sql = _clean_sql(response.content)
    return {"sql_query": sql}


# ── Node 5: generate_answer ──────────────────────────────────────────────
def generate_answer(state: TextToSQLState) -> dict:
    if state.get("error") and not state.get("sql_result"):
        return {"answer": f"[Could not answer — SQL error]: {state['error']}"}
    prompt = (
        "Answer the following question concisely based on the SQL result.\n\n"
        f"Question: {state['question']}\n"
        f"SQL result: {state['sql_result']}\n\n"
        "Answer:"
    )
    response = llm.invoke(prompt)
    return {"answer": response.content.strip()}


print("Step 1+2 applied: TABLE_GUIDE + 8 few-shot examples + SELECT column rule.")


Step 1+2 applied: TABLE_GUIDE + 9 few-shot examples + stronger SELECT-shape guard.


In [114]:
from langgraph.graph import StateGraph, START, END


def _route_after_execution(state: TextToSQLState) -> str:
    if state.get("error") and state.get("retry_count", 0) < 2:
        return "fix_sql"
    return "generate_answer"


builder = StateGraph(TextToSQLState)
builder.add_node("select_tables",   select_tables)
builder.add_node("generate_sql",    generate_sql)
builder.add_node("execute_sql",     execute_sql)
builder.add_node("fix_sql",         fix_sql)
builder.add_node("generate_answer", generate_answer)

builder.add_edge(START,            "select_tables")
builder.add_edge("select_tables",  "generate_sql")
builder.add_edge("generate_sql",   "execute_sql")
builder.add_conditional_edges(
    "execute_sql",
    _route_after_execution,
    {"fix_sql": "fix_sql", "generate_answer": "generate_answer"},
)
builder.add_edge("fix_sql",        "execute_sql")
builder.add_edge("generate_answer", END)

graph = builder.compile()
print("Graph compiled successfully.\n")
print(graph.get_graph().draw_mermaid())

Graph compiled successfully.

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	select_tables(select_tables)
	generate_sql(generate_sql)
	execute_sql(execute_sql)
	fix_sql(fix_sql)
	generate_answer(generate_answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> select_tables;
	execute_sql -.-> fix_sql;
	execute_sql -.-> generate_answer;
	fix_sql --> execute_sql;
	generate_sql --> execute_sql;
	select_tables --> generate_sql;
	generate_answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 4. Demo — Single Query

In [115]:
def run_query(question: str, verbose: bool = True) -> dict:
    """Run a natural-language question through the Text-to-SQL pipeline."""
    initial: TextToSQLState = {
        "question":        question,
        "relevant_tables": [],
        "schema":          "",
        "sql_query":       "",
        "sql_result":      None,
        "error":           None,
        "retry_count":     0,
        "answer":          "",
    }
    result = graph.invoke(initial)
    if verbose:
        print(f"Question : {result['question']}")
        print(f"Tables   : {result['relevant_tables']}")
        print(f"SQL      :\n  {result['sql_query']}")
        print(f"DB result: {result['sql_result']}")
        if result.get('error'):
            print(f"Error    : {result['error']}")
        print(f"Answer   : {result['answer']}")
    return result


# Demo
_ = run_query("Which club was founded first and in which year?")

Question : Which club was founded first and in which year?
Tables   : ['club']
SQL      :
  SELECT club_name, found_year FROM exp_v3.club ORDER BY found_year ASC LIMIT 1
DB result: [('R.S. Ginnastica Torino', 1844)]
Answer   : The club that was founded first is R.S. Ginnastica Torino, which was founded in 1844.


## 5. Test-Set Evaluation

**Metric: Execution Accuracy**  
A predicted query is counted as *correct* when its executed result set matches the gold result set exactly.  

**Test set:** 18 questions drawn from `data/dev.json` (Football DB, `exp_v3` schema).  
Split: 9 easy/medium + 9 hard/extra-hard (50 % / 50 %).


In [126]:
# 18 questions from data/dev.json (validation set — no overlap with training examples)
# Split: 9 easy/medium  |  9 hard/extra-hard
TEST_SET = [
    # ── EASY (3) ──────────────────────────────────────────────────────────
    {
        "id": 11, "hardness": "easy",
        "question": "At how many world cups did Italy participate in?",
        "gold_sql": "SELECT count(*) FROM national_team AS T1 WHERE T1.teamname = 'Italy'",
    },
    {
        "id": 144, "hardness": "easy",
        "question": "How many players have played so far?",
        "gold_sql": "SELECT count(distinct p.player_id) FROM player AS p JOIN player_fact AS pf ON p.player_id = pf.player_id",
    },
    {
        "id": 867, "hardness": "easy",
        "question": "Which teams played in world cup year 1972?",
        "gold_sql": "SELECT teamname FROM national_team WHERE year = 1972",
    },
    # ── MEDIUM (6) ────────────────────────────────────────────────────────
    {
        "id": 388, "hardness": "medium",
        "question": "What is the country code of Arrigo Sacchi?",
        "gold_sql": "SELECT country_code FROM coach WHERE coach_name ILIKE '%Arrigo Sacchi%'",
    },
    {
        "id": 491, "hardness": "medium",
        "question": "Which distinct players have gotten at least one yellow card at the world cup? Return the name.",
        "gold_sql": "SELECT DISTINCT t1.player_name FROM player AS t1 JOIN match_fact AS t2 ON t2.player_id = t1.player_id WHERE t2.yellow_card = 'true'",
    },
    {
        "id": 595, "hardness": "medium",
        "question": "Which teams are in group A in 2022?",
        "gold_sql": (
            "SELECT DISTINCT T2.teamname FROM plays_match AS T1 "
            "JOIN national_team AS T2 ON T1.team_id = T2.team_id "
            "WHERE T1.year = 2022 AND T1.stage = 'Group A'"
        ),
    },
    {
        "id": 928, "hardness": "medium",
        "question": "Who won the world cup 1994?",
        "gold_sql": (
            "SELECT T1.teamname FROM national_team AS T1 "
            "JOIN world_cup_result AS T2 ON T1.team_id = T2.team_id "
            "WHERE T2.year = 1994 AND T2.winner = 'true'"
        ),
    },
    {
        "id": 1024, "hardness": "medium",
        "question": "Total goal score of shirt number 13?",
        "gold_sql": (
            "SELECT count(*) FROM match_fact AS T1 "
            "JOIN player_fact AS T2 ON T1.player_id = T2.player_id "
            "WHERE T2.shirt_number = 13 AND T1.goal = 'true'"
        ),
    },
    {
        "id": 214, "hardness": "medium",
        "question": "List all clubs from Switzerland that were founded after 1930.",
        "gold_sql": "SELECT * FROM club WHERE country = 'Switzerland' AND found_year > 1930",
    },
    # ── HARD (5) ──────────────────────────────────────────────────────────
    {
        "id": 2, "hardness": "hard",
        "question": "Against which team did Switzerland lose in 2018?",
        "gold_sql": (
            "SELECT T1.teamname, T3.teamname, T2.team_goals, T2.opponent_team_goals "
            "FROM national_team AS T1 JOIN plays_match AS T2 ON T1.team_id = T2.team_id "
            "JOIN national_opponent_team AS T3 ON T2.opponent_team_id = T3.team_id "
            "WHERE T3.teamname = 'Switzerland' AND T2.year = 2018 AND T2.did_win = 'true'"
        ),
    },
    {
        "id": 122, "hardness": "hard",
        "question": "How many goals did Switzerland score in 2018?",
        "gold_sql": (
            "SELECT sum(T2.team_goals) FROM national_team AS T1 "
            "JOIN plays_match AS T2 ON T2.team_id = T1.team_id "
            "WHERE T1.teamname LIKE '%Switzerland%' AND T1.year = 2018"
        ),
    },
    {
        "id": 158, "hardness": "hard",
        "question": "How many times did Switzerland play against Brazil?",
        "gold_sql": (
            "SELECT count(*) FROM plays_match AS T1 "
            "JOIN national_team AS T2 ON T1.team_id = T2.team_id "
            "JOIN national_opponent_team AS T3 ON T1.opponent_team_id = T3.team_id "
            "WHERE T2.teamname = 'Switzerland' AND T3.teamname = 'Brazil'"
        ),
    },
    {
        "id": 327, "hardness": "hard",
        "question": "Show the result of the final of 2018",
        "gold_sql": (
            "SELECT T1.teamname, T2.team_goals FROM national_team AS T1 "
            "JOIN plays_match AS T2 ON T1.team_id = T2.team_id "
            "JOIN world_cup_result AS T3 ON T1.team_id = T3.team_id "
            "WHERE T2.stage = 'Final' AND T3.year = 2018"
        ),
    },
    {
        "id": 620, "hardness": "hard",
        "question": "Who did Germany draw against in the 2022 World Cup?",
        "gold_sql": (
            "SELECT T2.teamname FROM national_team AS T1 "
            "JOIN plays_match AS T3 ON T1.team_id = T3.team_id "
            "JOIN national_opponent_team AS T2 ON T3.opponent_team_id = T2.team_id "
            "WHERE T1.teamname = 'Germany' AND T2.year = 2022 AND T3.is_draw = 'true'"
        ),
    },
    # ── EXTRA-HARD (4) ────────────────────────────────────────────────────
    {
        "id": 438, "hardness": "extra",
        "question": "What was the score between Germany and Brazil in 2014?",
        "gold_sql": (
            "SELECT T1.teamname, T3.teamname, T2.team_goals, T2.opponent_team_goals "
            "FROM national_team AS T1 JOIN plays_match AS T2 ON T2.team_id = T1.team_id "
            "JOIN national_team AS T3 ON T3.team_id = T2.opponent_team_id "
            "WHERE T1.teamname LIKE '%Brazil%' AND T3.teamname LIKE '%Germany%' AND T2.year = 2014"
        ),
    },
    {
        "id": 518, "hardness": "extra",
        "question": "Which player has the most yellow cards overall?",
        "gold_sql": (
            "SELECT T1.player_name, count(T2.yellow_card) FROM player AS T1 "
            "JOIN match_fact AS T2 ON T1.player_id = T2.player_id "
            "GROUP BY T1.player_name ORDER BY count(T2.yellow_card) DESC LIMIT 1"
        ),
    },
    {
        "id": 565, "hardness": "extra",
        "question": "Which team has won the most championships?",
        "gold_sql": (
            "SELECT T1.teamname, count(*) FROM national_team AS T1 "
            "JOIN world_cup_result AS T2 ON T1.team_id = T2.team_id "
            "WHERE T2.winner = 'true' GROUP BY T1.teamname ORDER BY count(*) DESC LIMIT 1"
        ),
    },
    {
        "id": 691, "hardness": "extra",
        "question": "Who was the top scorer of the world cup 2018?",
        "gold_sql": (
            "SELECT T1.player_name, count(*) FROM player AS T1 "
            "JOIN match_fact AS T2 ON T1.player_id = T2.player_id "
            "JOIN plays_match AS T4 ON T2.match_team_id = T4.match_team_id "
            "JOIN national_team AS T3 ON T4.team_id = T3.team_id "
            "WHERE T2.goal = 'true' AND T3.year = 2018 "
            "GROUP BY T1.player_name ORDER BY count(*) DESC"
        ),
    },
]

print(f"Test set loaded: {len(TEST_SET)} questions")
by_hardness = {}
for q in TEST_SET:
    by_hardness.setdefault(q["hardness"], []).append(q["id"])
for h, ids in sorted(by_hardness.items()):
    print(f"  {h:8s}: {len(ids)} questions  (ids: {ids})")

Test set loaded: 18 questions
  easy    : 3 questions  (ids: [11, 144, 867])
  extra   : 4 questions  (ids: [438, 518, 565, 691])
  hard    : 5 questions  (ids: [2, 122, 158, 327, 620])
  medium  : 6 questions  (ids: [388, 491, 595, 928, 1024, 214])


In [127]:
def _normalize(result) -> str:
    """Normalise a SQL result for comparison (lower-case, strip whitespace)."""
    return str(result).strip().lower().replace(" ", "")


def evaluate(test_set: list, verbose: bool = True) -> dict:
    """
    Evaluate execution accuracy over test_set.

    For each item:
      1. Execute the gold SQL to obtain the reference result.
      2. Run the pipeline to obtain a predicted SQL and result.
      3. Mark as correct if the normalised results match.

    Returns a dict with 'accuracy', 'correct', 'total', and 'details'.
    """
    correct = 0
    details = []

    for i, item in enumerate(test_set):
        question = item["question"]
        gold_sql = item["gold_sql"]

        if verbose:
            print(f"\n[{i + 1}/{len(test_set)}] {question}")

        # Gold result
        try:
            gold_result = sqldb.run(gold_sql)
        except Exception as exc:
            gold_result = f"GOLD_ERROR: {exc}"

        # Pipeline prediction
        pred        = run_query(question, verbose=False)
        pred_sql    = pred.get("sql_query", "")
        pred_result = pred.get("sql_result")
        pred_error  = pred.get("error")

        match = _normalize(gold_result) == _normalize(pred_result)
        if match:
            correct += 1

        details.append({
            "question":         question,
            "gold_sql":         gold_sql,
            "gold_result":      gold_result,
            "predicted_sql":    pred_sql,
            "predicted_result": pred_result,
            "error":            pred_error,
            "correct":          match,
        })

        if verbose:
            status = "CORRECT" if match else "WRONG"
            print(f"  Gold SQL      : {gold_sql}")
            print(f"  Predicted SQL : {pred_sql}")
            print(f"  Gold result   : {gold_result}")
            print(f"  Pred result   : {pred_result}")
            if pred_error:
                print(f"  Error         : {pred_error}")
            print(f"  Status        : {status}")

    accuracy = correct / len(test_set) if test_set else 0.0
    return {"accuracy": accuracy, "correct": correct, "total": len(test_set), "details": details}


print("Evaluation function ready.")


Evaluation function ready.


In [128]:
eval_results = evaluate(TEST_SET, verbose=True)

print(f"\n{'=' * 55}")
print(f"Execution Accuracy: {eval_results['correct']}/{eval_results['total']} "
      f"= {eval_results['accuracy']:.1%}")
print(f"{'=' * 55}")



[1/18] At how many world cups did Italy participate in?
  Gold SQL      : SELECT count(*) FROM national_team AS T1 WHERE T1.teamname = 'Italy'
  Predicted SQL : SELECT COUNT(*) 
FROM exp_v3.national_team 
WHERE teamname = 'Italy';
  Gold result   : [(18,)]
  Pred result   : [(18,)]
  Status        : CORRECT

[2/18] How many players have played so far?
  Gold SQL      : SELECT count(distinct p.player_id) FROM player AS p JOIN player_fact AS pf ON p.player_id = pf.player_id
  Predicted SQL : SELECT COUNT(DISTINCT player_id) 
FROM exp_v3.player_fact;
  Gold result   : [(8775,)]
  Pred result   : [(8775,)]
  Status        : CORRECT

[3/18] Which teams played in world cup year 1972?
  Gold SQL      : SELECT teamname FROM national_team WHERE year = 1972
  Predicted SQL : SELECT DISTINCT T2.teamname
FROM exp_v3.plays_match AS T1
JOIN exp_v3.national_team AS T2 ON T1.team_id = T2.team_id
WHERE T1.year = 1972;
  Gold result   : 
  Pred result   : 
  Status        : CORRECT

[4/18] What is the 

In [129]:
import pandas as pd

df = pd.DataFrame([
    {
        "#":               i + 1,
        "Question":        r["question"],
        "Predicted SQL":   r["predicted_sql"],
        "Gold result":     r["gold_result"],
        "Predicted result":r["predicted_result"],
        "Correct":         "✓" if r["correct"] else "✗",
    }
    for i, r in enumerate(eval_results["details"])
])

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)
display(df)

n_correct = eval_results['correct']
n_total   = eval_results['total']
print(f"\nFinal Execution Accuracy: {n_correct}/{n_total} = {eval_results['accuracy']:.1%}")


,#,Question,Predicted SQL,Gold result,Predicted result,Correct
0,1,At how many world cups did Italy participate in?,SELECT COUNT(*) \nFROM exp_v3.national_team \nWHERE teamname = 'Italy';,"[(18,)]","[(18,)]",✓
1,2,How many players have played so far?,SELECT COUNT(DISTINCT player_id) \nFROM exp_v3.player_fact;,"[(8775,)]","[(8775,)]",✓
2,3,Which teams played in world cup year 1972?,SELECT DISTINCT T2.teamname\nFROM exp_v3.plays_match AS T1\nJOIN exp_v3.national_team AS T2 ON T...,,,✓
3,4,What is the country code of Arrigo Sacchi?,SELECT country_code \nFROM exp_v3.coach \nWHERE coach_name = 'Arrigo Sacchi';,"[('ITA',)]","[('ITA',)]",✓
4,5,Which distinct players have gotten at least one yellow card at the world cup? Return the name.,SELECT DISTINCT p.player_name\nFROM exp_v3.player AS p\nJOIN exp_v3.match_fact AS mf ON p.player...,"[('Davor Šuker',), ('Ulrich van Gobbel',), ('Luís Boa Morte',), ('Stefan de Vrij',), ('Alhassan ...","[('Davor Šuker',), ('Ulrich van Gobbel',), ('Luís Boa Morte',), ('Stefan de Vrij',), ('Alhassan ...",✓
5,6,Which teams are in group A in 2022?,SELECT DISTINCT T2.teamname\nFROM exp_v3.plays_match AS T1\nJOIN exp_v3.national_team AS T2 ON T...,"[('Ecuador',), ('Netherlands',), ('Qatar',), ('Senegal',)]","[('Ecuador',), ('Netherlands',), ('Qatar',), ('Senegal',)]",✓
6,7,Who won the world cup 1994?,SELECT T2.teamname\nFROM exp_v3.world_cup_result AS T1\nJOIN exp_v3.national_team AS T2 ON T1.te...,"[('Brazil',)]","[('Brazil',)]",✓
7,8,Total goal score of shirt number 13?,SELECT SUM(CASE WHEN mf.goal THEN 1 ELSE 0 END) AS total_goals\nFROM exp_v3.match_fact mf\nJOIN ...,"[(210,)]","[(None,)]",✗
8,9,List all clubs from Switzerland that were founded after 1930.,"SELECT club_name, found_year \nFROM exp_v3.club \nWHERE country = 'Switzerland' AND found_year >...","[('Q309456', 'FC Lausanne-Sport', 'Switzerland', 2003), ('Q335195', 'Neuchâtel Xamax', 'Switzerl...","[('FC Lausanne-Sport', 2003), ('Neuchâtel Xamax', 2013)]",✗
9,10,Against which team did Switzerland lose in 2018?,SELECT T2.teamname\nFROM exp_v3.plays_match AS T1\nJOIN exp_v3.national_team AS T2 ON T1.opponen...,"[('Switzerland', 1, 0)]","[('Sweden',)]",✗



Final Execution Accuracy: 9/18 = 50.0%
